In [ ]:
from google.colab import files

print("Please upload the 'cpc_cache.zip' file.")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


In [ ]:
import zipfile
import os
# Extract the uploaded zip
with zipfile.ZipFile('cpc_cache.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
# Check what we have
# Using a raw string and backslashes to match the extracted file paths.
scheme_files = glob.glob(r'/content/cpc_scheme_2026\cpc-scheme-*.xml')
print(f"✅ Found {len(scheme_files)} scheme files")
print(f"Sample files:")
for f in sorted(scheme_files)[:5]:
    print(f"  - {os.path.basename(f)}")

In [ ]:
# Cell 1: Install dependencies\n
!pip install networkx sentence-transformers numpy scikit-learn lxml -q\n
print("✅ Dependencies installed")

In [ ]:
import os
import sys
import json
import pickle
import hashlib
import logging
import glob
from lxml import etree as ET
from typing import Dict, List, Optional
from datetime import datetime
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)
# ============================================================================
# FIX 1: Extract zip if needed
# ============================================================================
import zipfile
if os.path.exists('/content/cpc_scheme_2026'):
    print("✅ Found cpc_scheme_2026 folder")
else:
    print("Looking for cpc_cache.zip...")
    if os.path.exists('/content/cpc_cache.zip'):
        print("Extracting zip...")
        with zipfile.ZipFile('/content/cpc_cache.zip', 'r') as z:
            z.extractall('/content/')
        print("✅ Extracted!")
    else:
        print("❌ ERROR: No cpc_scheme_2026 folder or cpc_cache.zip found!")
        print("Please upload your files first")
        sys.exit(1)
# ============================================================================
# FIX 2: Find XML files (handles Windows backslash paths)
# ============================================================================
# Use the proven working pattern to find XML files
xml_files = glob.glob(r'/content/cpc_scheme_2026\cpc-scheme-*.xml')

print(f"\n✅ Found {len(xml_files)} XML files")
if len(xml_files) > 0:
    print("Sample files:")
    for f in sorted(xml_files)[:5]:
        print(f"  - {os.path.basename(f)}")
else:
    print("❌ No XML files found!")
    print("Directory contents:")
    for root, dirs, files in os.walk('/content/cpc_scheme_2026'):
        for f in files[:10]:
            print(f"  {os.path.join(root, f)}")
    sys.exit(1)
# ============================================================================
# FIX 3: Test XML parsing with namespace
# ============================================================================
print("\n" + "="*60)
print("TESTING XML PARSING")
print("="*60)
test_file = xml_files[0]
tree = ET.parse(test_file)
root = tree.getroot()
# Test with namespace
ns_items = list(root.iter('{http://www.epo.org/cpc}classification-item'))
print(f"With namespace: {len(ns_items)} items")
# Test without namespace
no_ns_items = list(root.iter('classification-item'))
print(f"Without namespace: {len(no_ns_items)} items")
if ns_items:
    item = ns_items[0]
    sym = item.find('{http://www.epo.org/cpc}classification-symbol')
    title = item.find('{http://www.epo.org/cpc}class-title')
    sym_text = sym.text if sym is not None else 'N/A'
    title_element_text = title.text if title is not None else None
    title_display_text = title_element_text[:50] if title_element_text is not None else 'N/A'
    print(f"Sample: {sym_text} - {title_display_text}...")
elif no_ns_items:
    item = no_ns_items[0]
    sym = item.find('classification-symbol')
    title = item.find('class-title')
    sym_text = sym.text if sym is not None else 'N/A'
    title_element_text = title.text if title is not None else None
    title_display_text = title_element_text[:50] if title_element_text is not None else 'N/A'
    print(f"Sample: {sym_text} - {title_display_text}...")
else:
    print("❌ No classification items found in XML!")
    print(f"Root tag: {root.tag}")
    print(f"Root attrib: {root.attrib}")
    sys.exit(1)
# ============================================================================
# PART 4: Build Knowledge Graph
# ============================================================================
class CPCKnowledgeGraph:
    def __init__(self, cache_dir: str, model_name: str = 'all-mpnet-base-v2'):
        self.cache_dir = cache_dir
        self.model_name = model_name
        self.model = None
        self.graph = nx.DiGraph()
        self.embeddings = {}
        self.symbol_texts = {}
        self.class_to_subgroups = {}
        self.source_hash = ""
    def _get_model(self):
        if self.model is None:
            logger.info('Loading embedding model: %s (110MB)...', self.model_name)
            self.model = SentenceTransformer(self.model_name)
            logger.info('✅ Model loaded!')
        return self.model
    def build_from_xml(self, xml_files: List[str]):
        logger.info('=' * 60)
        logger.info('CPC Knowledge Graph Builder')
        logger.info('Started: %s', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
        logger.info('=' * 60)
        logger.info('Processing %d XML files...', len(xml_files))
        # Parse all XML files
        all_subgroups = []
        for i, xml_file in enumerate(xml_files):
            try:
                subgroups = self._parse_xml_file(xml_file)
                all_subgroups.extend(subgroups)
                if (i + 1) % 50 == 0 or i == len(xml_files) - 1:
                    logger.info('  Parsed %d/%d files (%d subgroups)', i + 1, len(xml_files), len(all_subgroups))
            except Exception as e:
                logger.warning('Failed to parse %s: %s', os.path.basename(xml_file), e)
        logger.info('Total subgroups extracted: %d', len(all_subgroups))

        if len(all_subgroups) == 0:
            logger.error("❌ ERROR: No subgroups extracted!")
            return
        # Build graph and embeddings
        self._build_graph_structure(all_subgroups)
        self._generate_embeddings()
        # Save
        self.save()
        logger.info('=' * 60)
        logger.info('✅ Build Complete!')
        logger.info('Finished: %s', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
        logger.info('=' * 60)
    def _parse_xml_file(self, xml_file: str) -> List[dict]:
        """Parse a single CPC scheme XML file."""
        tree = ET.parse(xml_file)
        root = tree.getroot()
        subgroups = []

        # Try with namespace first
        items = list(root.iter('{http://www.epo.org/cpc}classification-item'))
        use_ns = True

        # Fallback without namespace
        if not items:
            items = list(root.iter('classification-item'))
            use_ns = False

        if not items:
            return subgroups
        ns_prefix = '{http://www.epo.org/cpc}' if use_ns else ''
        for item in items:
            symbol_elem = item.find(f'{ns_prefix}classification-symbol')
            title_elem = item.find(f'{ns_prefix}class-title')
            if symbol_elem is None or title_elem is None:
                continue
            symbol = symbol_elem.text or ''
            title = ''
            # Extract title text
            for text_elem in title_elem.iter():
                if text_elem.text:
                    title += ' ' + text_elem.text
                if text_elem.tail:
                    title += ' ' + text_elem.tail
            title = title.strip()
            if not symbol or not title:
                continue
            # Get parent chain
            parent_chain = []
            parent = item.getparent()
            while parent is not None:
                parent_symbol = parent.find(f'{ns_prefix}classification-symbol')
                if parent_symbol is not None and parent_symbol.text:
                    parent_chain.insert(0, parent_symbol.text)
                parent = parent.getparent()
            is_allocatable = item.get('allocatable', 'false').lower() == 'true'
            subgroups.append({
                'symbol': symbol,
                'title': title,
                'level': len(parent_chain),
                'parent_chain': parent_chain,
                'is_allocatable': is_allocatable
            })
        return subgroups
    def _build_graph_structure(self, subgroups):
        logger.info('Building graph structure...')

        # Fast lookup
        logger.info('  Creating symbol index...')
        symbol_to_title = {}
        for sg in subgroups:
            sym = sg.get('symbol', '')
            if sym:
                symbol_to_title[sym] = sg.get('title', '').strip()
        logger.info('  Symbol index: %d entries', len(symbol_to_title))
        char3_classes = set()
        total = len(subgroups)
        last_progress = 0
        logger.info('  Processing subgroups...')
        for idx, sg in enumerate(subgroups):
            symbol = sg.get('symbol', '')
            title = sg.get('title', '').strip()
            level = sg.get('level', 0)
            parent_chain = sg.get('parent_chain', [])
            is_allocatable = sg.get('is_allocatable', False)
            if not symbol or not title:
                continue
            prefix3 = symbol[:3] if len(symbol) >= 3 else symbol
            char3_classes.add(prefix3)
            # Add node
            self.graph.add_node(
                symbol,
                type='subgroup',
                title=title,
                level=level,
                is_allocatable=is_allocatable,
                prefix3=prefix3
            )
            # Build context text
            context_parts = []
            for parent in parent_chain:
                parent_title = symbol_to_title.get(parent, '')
                if parent_title:
                    context_parts.append(parent_title)
            context_parts.append(title)
            full_text = ' | '.join(filter(None, context_parts))
            self.symbol_texts[symbol] = full_text
            # Map to class
            if prefix3 not in self.class_to_subgroups:
                self.class_to_subgroups[prefix3] = []
            self.class_to_subgroups[prefix3].append(symbol)
            # Hierarchy edges
            if parent_chain:
                immediate_parent = parent_chain[-1]
                self.graph.add_edge(immediate_parent, symbol, relation='parent_of', weight=1.0)
            # Progress
            progress = int((idx + 1) / total * 100)
            if progress >= last_progress + 10:
                logger.info('    Progress: %d%% (%d/%d)', progress, idx + 1, total)
                last_progress = progress
        # Add class nodes
        for cls in char3_classes:
            self.graph.add_node(cls, type='class_3char', title=f'CPC Class {cls}', level=2)
            for sg_symbol in self.class_to_subgroups.get(cls, []):
                self.graph.add_edge(cls, sg_symbol, relation='contains', weight=0.5)
        logger.info('Graph built: %d nodes, %d edges',
                   self.graph.number_of_nodes(), self.graph.number_of_edges())
    def _generate_embeddings(self):
        logger.info('Generating embeddings (10-15 min on GPU)...')
        model = self._get_model()
        symbols = list(self.symbol_texts.keys())
        texts = [self.symbol_texts[s] for s in symbols]
        batch_size = 256
        all_embeddings = []
        total_batches = (len(texts) + batch_size - 1) // batch_size
        logger.info('  Processing %d subgroups in %d batches...', len(texts), total_batches)
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            batch_embeddings = model.encode(
                batch_texts,
                show_progress_bar=False,
                convert_to_numpy=True
            )
            all_embeddings.append(batch_embeddings)
            batch_num = i // batch_size + 1
            if batch_num % max(1, total_batches // 10) == 0 or batch_num == total_batches:
                logger.info('    Batch %d/%d (%.0f%%)', batch_num, total_batches,
                           batch_num / total_batches * 100)
        all_embeddings = np.vstack(all_embeddings)
        for idx, symbol in enumerate(symbols):
            self.embeddings[symbol] = all_embeddings[idx]
        logger.info('Embeddings: %d vectors of %d dimensions',
                   len(self.embeddings), all_embeddings.shape[1])
    def save(self):
        os.makedirs(self.cache_dir, exist_ok=True)
        logger.info('Saving graph...')
        graph_path = os.path.join(self.cache_dir, 'cpc_graph.pkl')
        with open(graph_path, 'wb') as f:
            pickle.dump(self.graph, f)
        logger.info('Saving embeddings...')
        embeddings_path = os.path.join(self.cache_dir, 'cpc_embeddings.npz')
        np.savez_compressed(
            embeddings_path,
            symbols=list(self.embeddings.keys()),
            embeddings=np.array(list(self.embeddings.values()))
        )
        meta_path = os.path.join(self.cache_dir, 'cpc_graph_meta.json')
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump({
                'source_hash': self.source_hash,
                'num_nodes': self.graph.number_of_nodes(),
                'num_edges': self.graph.number_of_edges(),
                'num_embeddings': len(self.embeddings),
                'model_name': self.model_name,
                'build_date': datetime.now().isoformat()
            }, f, indent=2)
        logger.info('✅ Saved to: %s', self.cache_dir)
        logger.info('  - cpc_graph.pkl')
        logger.info('  - cpc_embeddings.npz')
        logger.info('  - cpc_graph_meta.json')
# ============================================================================
# RUN THE BUILDER
# ============================================================================
print("\n" + "=" * 60)
print("STARTING BUILD")
print("=" * 60)
graph = CPCKnowledgeGraph(cache_dir='/content/kg_cache')
graph.build_from_xml(xml_files)
print("\n" + "=" * 60)
print("BUILD COMPLETE!")
print("=" * 60)
# ============================================================================
# DOWNLOAD RESULTS
# ============================================================================
from google.colab import files
print("\nPreparing downloads...")
for fname in ['cpc_graph.pkl', 'cpc_embeddings.npz', 'cpc_graph_meta.json']:
    fpath = f'/content/kg_cache/{fname}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✅ {fname}: {size_mb:.1f} MB")
    else:
        print(f"  ❌ {fname}: NOT FOUND")
print("\nDownloading...")
files.download('/content/kg_cache/cpc_graph.pkl')
files.download('/content/kg_cache/cpc_embeddings.npz')
files.download('/content/kg_cache/cpc_graph_meta.json')
print("\n✅ DONE! Copy these files to your local project:")
print("  patent_cpc_fastapi/app/cpc_classification/resources/")